# Encodeur Transformer sur les métriques — script retenu

Encodeur entraîné de zéro (PyTorch) : projection LazyLinear vers 40 dimensions, 5 couches, 4 têtes, dropout 0,3, AdamW (lr 1e-5), 100 époques, batch 32. Standardisation transductive (seuil σ ≤ 1e-5), variantes SMOTE, sous-échantillonnage et valeurs aberrantes. Le notebook imprime 872 682 paramètres.

**Statut :** Retenu (tableau 11)

*Mémoire de maîtrise — Mireille Kafando, UQTR · 2025*

> Notebook de recherche conçu pour Google Colab. Les chemins `/content/drive/MyDrive/bug/...` sont à adapter (voir `data/README.md`). Les sorties ont été retirées.


# Mounting drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gmean
from imblearn.under_sampling import RandomUnderSampler

class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 100
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 40
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2

class BugDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

class TransModel(nn.Module):
    def __init__(self):
        super(TransModel, self).__init__()

        self.embedding = Config.EMBEDDING_DIM

        # Utilisation de LazyLinear au lieu de Linear
        self.enc_embedding = nn.Sequential(
            nn.LazyLinear(self.embedding, bias=False),
            nn.LazyLinear(self.embedding, bias=False),
            nn.LayerNorm(self.embedding)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        self.encoder = nn.Sequential(
            self.enc_embedding,
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=Config.NUM_LAYERS,
                norm=nn.LayerNorm(self.embedding)
            )
        )

        # Utilisation de LazyLinear
        self.classifier = nn.Sequential(
            nn.LazyLinear(4 * self.embedding),
            nn.ReLU(),
            nn.LazyLinear(Config.NUM_CLASSES),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        encoded = self.encoder(x)
        encoded = encoded.squeeze(1)
        return self.classifier(encoded)

def load_and_preprocess_data(train_files, val_file):
    # Load and concatenate training data
    train_dfs = []
    for file in train_files:
        df = pd.read_csv(file, usecols=lambda x: x != 'name')
        train_dfs.append(df)
    df_train = pd.concat(train_dfs, ignore_index=True)

    # Load validation data
    df_val = pd.read_csv(val_file, usecols=lambda x: x != 'name')

    # Convert to binary labels
    df_train['bug'] = (df_train['bug'] > 0).astype(int)
    df_val['bug'] = (df_val['bug'] > 0).astype(int)

    # Custom standardization for training data
    feature_cols = [col for col in df_train.columns if col != 'bug']
    means_tr = df_train[feature_cols].mean(axis=0)
    stds_tr = df_train[feature_cols].std(axis=0)

    # Custom standardization for validation data
    means_val = df_val[feature_cols].mean(axis=0)
    stds_val = df_val[feature_cols].std(axis=0)

    # Remove columns with small standard deviation
    cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-5].index.values
    cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values

    # Drop columns with small std
    df_train = df_train.drop(cols_to_drop_training, axis=1)
    df_val = df_val.drop(cols_to_drop_val, axis=1)

    # Get remaining feature columns
    feature_cols_train = [col for col in df_train.columns if col != 'bug']
    feature_cols_val = [col for col in df_val.columns if col != 'bug']

    # Standardize remaining features
    df_train[feature_cols_train] = (df_train[feature_cols_train] - means_tr[feature_cols_train]) / stds_tr[feature_cols_train]
    df_val[feature_cols_val] = (df_val[feature_cols_val] - means_val[feature_cols_val]) / stds_val[feature_cols_val]

    # Apply RandomUnderSampler to balance the training data by undersampling the majority class
    rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
    X_train_resampled, y_train_resampled = rus.fit_resample(
        df_train.iloc[:, :-1].values,
        df_train.iloc[:, -1:].values
    )

    # Convert back to DataFrames to maintain column names
    x_train = pd.DataFrame(X_train_resampled, columns=df_train.columns[:-1])
    y_train = pd.DataFrame(y_train_resampled, columns=['bug'])
    df_training = pd.concat([x_train, y_train], axis=1)

    # Prepare final numpy arrays for PyTorch
    X_train_final = df_training.iloc[:, :-1].values
    y_train_final = df_training.iloc[:, -1].values
    X_val_final = df_val[feature_cols_val].values
    y_val_final = df_val['bug'].values

    print(f"Training samples before undersampling: {len(df_train)}")
    print(f"Training samples after undersampling: {len(df_training)}")
    print(f"Features in training set: {X_train_final.shape[1]}")
    print(f"Features in validation set: {X_val_final.shape[1]}")

    return X_train_final, y_train_final, X_val_final, y_val_final

def calculate_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    g_mean = gmean([sensitivity, specificity])

    return cm, sensitivity, specificity, g_mean

def plot_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(7, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def train_and_validate():
    train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.7.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-3.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.7.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',

    ]
    val_file_path = '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv'
    X_train, y_train, X_val, y_val = load_and_preprocess_data(train_file_paths, val_file_path)

    train_dataset = BugDataset(X_train, y_train)
    val_dataset = BugDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    model = TransModel().to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for features, labels in train_loader:
            features = features.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print("-" * 50)

    cm, sensitivity, specificity, g_mean = calculate_metrics(all_labels, all_preds)

    plt.figure(figsize=(7, 3))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(cm)

    print("\nFinal Metrics:")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"G-mean: {g_mean:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

if __name__ == "__main__":
    train_and_validate()

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gmean
from imblearn.over_sampling import SMOTE

class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 100
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 40
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2

class BugDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

class TransModel(nn.Module):
    def __init__(self):
        super(TransModel, self).__init__()

        self.embedding = Config.EMBEDDING_DIM

        # Utilisation de LazyLinear au lieu de Linear
        self.enc_embedding = nn.Sequential(
            nn.LazyLinear(self.embedding, bias=False),
            nn.LazyLinear(self.embedding, bias=False),
            nn.LayerNorm(self.embedding)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        self.encoder = nn.Sequential(
            self.enc_embedding,
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=Config.NUM_LAYERS,
                norm=nn.LayerNorm(self.embedding)
            )
        )

        # Utilisation de LazyLinear
        self.classifier = nn.Sequential(
            nn.LazyLinear(4 * self.embedding),
            nn.ReLU(),
            nn.LazyLinear(Config.NUM_CLASSES),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        encoded = self.encoder(x)
        encoded = encoded.squeeze(1)
        return self.classifier(encoded)

def load_and_preprocess_data(train_files, val_file):
    # Load and concatenate training data
    train_dfs = []
    for file in train_files:
        df = pd.read_csv(file, usecols=lambda x: x != 'name')
        train_dfs.append(df)
    df_train = pd.concat(train_dfs, ignore_index=True)

    # Load validation data
    df_val = pd.read_csv(val_file, usecols=lambda x: x != 'name')

    # Convert to binary labels
    df_train['bug'] = (df_train['bug'] > 0).astype(int)
    df_val['bug'] = (df_val['bug'] > 0).astype(int)

    # Custom standardization for training data
    feature_cols = [col for col in df_train.columns if col != 'bug']
    means_tr = df_train[feature_cols].mean(axis=0)
    stds_tr = df_train[feature_cols].std(axis=0)

    # Custom standardization for validation data
    means_val = df_val[feature_cols].mean(axis=0)
    stds_val = df_val[feature_cols].std(axis=0)

    # Remove columns with small standard deviation
    cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-5].index.values
    cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values

    # Drop columns with small std
    df_train = df_train.drop(cols_to_drop_training, axis=1)
    df_val = df_val.drop(cols_to_drop_val, axis=1)

    # Get remaining feature columns
    feature_cols_train = [col for col in df_train.columns if col != 'bug']
    feature_cols_val = [col for col in df_val.columns if col != 'bug']

    # Standardize remaining features
    df_train[feature_cols_train] = (df_train[feature_cols_train] - means_tr[feature_cols_train]) / stds_tr[feature_cols_train]
    df_val[feature_cols_val] = (df_val[feature_cols_val] - means_val[feature_cols_val]) / stds_val[feature_cols_val]

    # Apply SMOTE to balance the training data
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(
        df_train.iloc[:, :-1].values,
        df_train.iloc[:, -1:].values
    )

    # Convert back to DataFrames to maintain column names
    x_train = pd.DataFrame(X_train_resampled, columns=df_train.columns[:-1])
    y_train = pd.DataFrame(y_train_resampled, columns=['bug'])
    df_training = pd.concat([x_train, y_train], axis=1)

    # Prepare final numpy arrays for PyTorch
    X_train_final = df_training.iloc[:, :-1].values
    y_train_final = df_training.iloc[:, -1].values
    X_val_final = df_val[feature_cols_val].values
    y_val_final = df_val['bug'].values

    print(f"Training samples before SMOTE: {len(df_train)}")
    print(f"Training samples after SMOTE: {len(df_training)}")
    print(f"Features in training set: {X_train_final.shape[1]}")
    print(f"Features in validation set: {X_val_final.shape[1]}")

    return X_train_final, y_train_final, X_val_final, y_val_final

def calculate_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    g_mean = gmean([sensitivity, specificity])

    return cm, sensitivity, specificity, g_mean

def plot_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(7, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def train_and_validate():
    train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.7.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-3.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.7.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv',


    ]
    val_file_path = '/content/drive/MyDrive/bug/bug-data/camel/camel-1.6.csv'

    X_train, y_train, X_val, y_val = load_and_preprocess_data(train_file_paths, val_file_path)

    train_dataset = BugDataset(X_train, y_train)
    val_dataset = BugDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    model = TransModel().to(Config.DEVICE)

    # ✅ AJOUTER ICI
    print("\n" + "="*80)
    print("VÉRIFICATION DU NOMBRE DE PARAMÈTRES")
    print("="*80)

    # Initialiser avec le bon nombre de features
    num_features = X_train.shape[1]  # Utiliser le vrai nombre de features
    dummy_input = torch.randn(1, num_features).to(Config.DEVICE)
    _ = model(dummy_input)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nNombre total de paramètres: {total_params:,}")
    print("="*80 + "\n")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for features, labels in train_loader:
            features = features.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print("-" * 50)

    cm, sensitivity, specificity, g_mean = calculate_metrics(all_labels, all_preds)

    plt.figure(figsize=(7, 3))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(cm)

    print("\nFinal Metrics:")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"G-mean: {g_mean:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

if __name__ == "__main__":
    train_and_validate()

In [ ]:
import torch
import torch.nn as nn

class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EMBEDDING_DIM = 40
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2

class TransModel(nn.Module):
    def __init__(self):
        super(TransModel, self).__init__()
        self.embedding = Config.EMBEDDING_DIM

        self.enc_embedding = nn.Sequential(
            nn.LazyLinear(self.embedding, bias=False),
            nn.LazyLinear(self.embedding, bias=False),
            nn.LayerNorm(self.embedding)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        self.encoder = nn.Sequential(
            self.enc_embedding,
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=Config.NUM_LAYERS,
                norm=nn.LayerNorm(self.embedding)
            )
        )

        self.classifier = nn.Sequential(
            nn.LazyLinear(4 * self.embedding),
            nn.ReLU(),
            nn.LazyLinear(Config.NUM_CLASSES),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        encoded = self.encoder(x)
        encoded = encoded.squeeze(1)
        return self.classifier(encoded)

def analyze_model():
    """Analyse complète du modèle"""

    model = TransModel().to(Config.DEVICE)

    # ÉTAPE CRITIQUE: Initialiser avec 20 features
    dummy_input = torch.randn(1, 20).to(Config.DEVICE)
    output = model(dummy_input)

    print(f"\n{'='*80}")
    print(f"ANALYSE DU MODÈLE TRANSFORMER")
    print(f"{'='*80}")
    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"{'='*80}\n")

    # Compter les paramètres par module
    print(f"{'COUCHE':<50} {'PARAMÈTRES':>20}")
    print(f"{'-'*70}")

    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            n_params = param.numel()
            total += n_params
            print(f"{name:<50} {n_params:>20,}")

    print(f"{'-'*70}")
    print(f"{'TOTAL':<50} {total:>20,}")
    print(f"{'='*80}\n")

    # Grouper par sous-modules
    print(f"{'='*80}")
    print(f"RÉPARTITION PAR SOUS-MODULE")
    print(f"{'='*80}")

    embedding_params = sum(p.numel() for n, p in model.named_parameters()
                          if 'enc_embedding' in n and p.requires_grad)
    encoder_params = sum(p.numel() for n, p in model.named_parameters()
                        if 'encoder.1' in n and p.requires_grad)
    classifier_params = sum(p.numel() for n, p in model.named_parameters()
                           if 'classifier' in n and p.requires_grad)

    print(f"Embedding layers:      {embedding_params:>15,} params")
    print(f"Transformer Encoder:   {encoder_params:>15,} params")
    print(f"Classifier:            {classifier_params:>15,} params")
    print(f"{'-'*80}")
    print(f"TOTAL:                 {total:>15,} params")
    print(f"{'='*80}\n")

    return total

if __name__ == "__main__":
    total_params = analyze_model()

    print(f"\n🎯 RÉSULTAT FINAL: {total_params:,} paramètres\n")

In [ ]:
def train_and_validate():
    # ... (votre code existant) ...

    model = TransModel().to(Config.DEVICE)

    # ✅ AJOUTER ICI
    print("\n" + "="*80)
    print("VÉRIFICATION DU NOMBRE DE PARAMÈTRES")
    print("="*80)

    # Initialiser avec le bon nombre de features
    num_features = X_train.shape[1]  # Utiliser le vrai nombre de features
    dummy_input = torch.randn(1, num_features).to(Config.DEVICE)
    _ = model(dummy_input)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nNombre total de paramètres: {total_params:,}")
    print("="*80 + "\n")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
